# Implement and manage data quality constraints with Azure Databricks

## Industry Context: Insurance Claims Processing

**InsureCo** is a mid-size insurance carrier processing claims from broker networks across Auto, Property, Life, Health, and Liability lines.

Broker systems submit claim data in varying formats and quality levels. Without rigorous data quality controls at ingestion, downstream actuarial models, fraud detection pipelines, and regulatory reports receive corrupted records — resulting in incorrect reserve calculations and compliance violations.

This demo walks through how **Lakeflow Spark Declarative Pipelines** and **Delta Lake constraints** enforce data quality at every layer of the pipeline.

### Demo Setup

Before starting, run the setup function to create the insurance demo data in `trainer_demo.demo_09`:

In [ ]:
from scripts.setup import setup, setup_09

spark.sql("USE CATALOG trainer_demo")
setup_09(spark)

**Tables created:**
| Table | Description |
|---|---|
| `trainer_demo.demo_09.policies` | Clean reference policy data (300 records) |
| `trainer_demo.demo_09.raw_claims` | Raw claims with intentional data quality issues (~515 records) |
| `trainer_demo.demo_09.raw_policies` | Raw policies for schema drift demo (200 records) |

**Pipeline files** are located in `Demo/pipelines/09-data-quality/`:
| File | Demonstrates |
|---|---|
| `01_bronze_claims.py` | Nullability, cardinality, range checks |
| `02_type_checks.py` | Data type validation and quarantine pattern |
| `03_schema_drift.py` | Schema drift detection and error handling |
| `04_pipeline_expectations.py` | Full expectations framework |

## Implement validation checks

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/implement-manage-data-quality-constraints-unity-catalog.implement-validation-checks.png)

### Demo: Implement Validation Checks

**Scenario**: InsureCo's raw claims arrive from brokers with nulls, duplicates, and out-of-range amounts.

**Pipeline file**: `Demo/pipelines/09-data-quality/01_bronze_claims.py`

**Steps**:
1. Explore the raw claims data and identify quality issues
2. Show `WARN` action — all records flow through, violations logged
3. Show `DROP` action — null claim_ids and policy_ids removed
4. Show `RANGE` checks — negative and extreme amounts dropped
5. Show `FAIL` action — pipeline stops for approved claims without an adjuster
6. Check cardinality — view the duplicate claim ID report

> Create a **Lakeflow Spark Declarative Pipeline**, add `01_bronze_claims.py` as a source file,
> set the target catalog to `trainer_demo`, schema to `demo_09`, then run in **Development** mode.

In [ ]:
%sql
-- Explore raw claims data - understand the quality issues before the pipeline run

USE CATALOG trainer_demo;
USE SCHEMA demo_09;

-- Count quality violations across all validation dimensions
SELECT
  COUNT(*) AS total_records,
  COUNT_IF(claim_id IS NULL) AS null_claim_ids,
  COUNT_IF(policy_id IS NULL) AS null_policy_ids,
  COUNT_IF(claim_amount <= 0) AS negative_or_zero_amounts,
  COUNT_IF(claim_amount > 5000000) AS amounts_over_5m,
  COUNT_IF(claim_date > current_date()) AS future_dated_claims,
  COUNT_IF(claim_type NOT IN ('Auto', 'Property', 'Life', 'Health', 'Liability')) AS invalid_claim_types,
  COUNT_IF(status IS NULL) AS null_statuses,
  COUNT_IF(status = 'Approved' AND adjuster_id IS NULL) AS approved_no_adjuster
FROM trainer_demo.demo_09.raw_claims;

In [ ]:
%sql
-- Cardinality check — identify duplicate claim submissions (from the unit)

SELECT
  claim_id,
  COUNT(*) AS submission_count
FROM trainer_demo.demo_09.raw_claims
WHERE claim_id IS NOT NULL
GROUP BY claim_id
HAVING COUNT(*) > 1
ORDER BY submission_count DESC
LIMIT 20;

In [ ]:
# After running pipeline 01_bronze_claims.py — compare record counts
# The differences show how many records each validation stage dropped

for table, desc in [
    ("raw_claims",               "Raw input"),
    ("bronze_claims_warn",        "WARN: all records kept, violations logged"),
    ("bronze_claims_nullable",    "DROP: nulls on claim_id / policy_id removed"),
    ("bronze_claims_range_checked","DROP: nulls + range + date violations removed"),
    ("bronze_claims_duplicates",  "Duplicate claim IDs (cardinality report)"),
]:
    try:
        cnt = spark.sql(f"SELECT COUNT(*) FROM trainer_demo.demo_09.{table}").collect()[0][0]
        print(f"  {table:<38} {cnt:>5}  {desc}")
    except:
        print(f"  {table:<38}  N/A  (run the pipeline first)")

In [ ]:
%sql
-- Delta Lake table constraints — enforce validation rules outside the pipeline
-- Demonstrates NOT NULL and CHECK constraints from the unit
-- Create a clean target table with constraints
CREATE OR REPLACE TABLE trainer_demo.demo_09.claims_with_constraints (
  claim_id INT NOT NULL,
  policy_id STRING NOT NULL,
  claim_amount DOUBLE,
  claim_date DATE,
  claim_type STRING,
  CONSTRAINT positive_claim_amount CHECK (claim_amount > 0),
  CONSTRAINT max_claim_amount CHECK (claim_amount <= 5000000),
  CONSTRAINT valid_claim_type CHECK (
    claim_type IN ('Auto', 'Property', 'Life', 'Health', 'Liability')
  )
);

In [ ]:
%sql
-- Valid insert — should succeed
INSERT INTO trainer_demo.demo_09.claims_with_constraints
VALUES (1, 'POL-00001', 15000.00, current_date(), 'Auto');

In [ ]:
%sql
-- Invalid insert — should fail on NOT NULL

INSERT INTO trainer_demo.demo_09.claims_with_constraints
VALUES (NULL, 'POL-00002', 5000.00, current_date(), 'Auto');

In [ ]:
%sql
-- Invalid insert — should fail on CHECK constraint

INSERT INTO trainer_demo.demo_09.claims_with_constraints
VALUES (2, 'POL-00003', -500.00, current_date(), 'Auto');

## Implement data type checks

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/implement-manage-data-quality-constraints-unity-catalog.implement-data-type-checks.png)

### Demo: Implement Data Type Checks

**Scenario**: InsureCo receives claim amounts and dates as raw strings from legacy broker feeds.
Type mismatches break downstream actuarial calculations.

**Pipeline file**: `Demo/pipelines/09-data-quality/02_type_checks.py`

**Steps**:
1. Show Delta Lake schema enforcement — try inserting incompatible types
2. Demonstrate `TRY_CAST` to classify valid vs invalid records
3. Add a REGEXP CHECK constraint to validate numeric string columns
4. Run the pipeline and compare `silver_valid_claims` vs `bronze_quarantine_claims`

In [ ]:
%sql

-- 1. Schema enforcement — Delta Lake blocks type-incompatible writes automatically
INSERT INTO trainer_demo.demo_09.policies
VALUES (
  'POL-BAD-001',
  999,
  'Auto',
  '2025-01-01',
  '2026-01-01',
  'not-a-number',
  50000.0
);

In [ ]:
%sql

-- 2. Use TRY_CAST to identify type problems — records where result is NULL are invalid

SELECT
  claim_id,
  claim_amount,
  TRY_CAST(claim_amount AS DECIMAL(12,2)) AS validated_amount,
  CASE
    WHEN TRY_CAST(claim_amount AS DECIMAL(12,2)) IS NULL THEN 'Cannot cast to decimal'
    WHEN claim_amount <= 0 THEN 'Non-positive'
    WHEN claim_amount > 5000000 THEN 'Exceeds limit'
    ELSE 'Valid'
  END AS amount_status,
  claim_date,
  TRY_CAST(claim_date AS DATE) AS validated_date
FROM trainer_demo.demo_09.raw_claims
ORDER BY amount_status
LIMIT 20;

In [ ]:
%sql
-- 3. CHECK constraint with REGEXP — validate that a string column looks like a number

CREATE OR REPLACE TABLE trainer_demo.demo_09.claims_amount_string (
  claim_id INT NOT NULL,
  policy_id STRING NOT NULL,
  raw_amount STRING,
  CONSTRAINT numeric_amount CHECK (raw_amount REGEXP '^[0-9]+(\\.[0-9]+)?$')
);

-- Valid numeric string — should succeed
INSERT INTO trainer_demo.demo_09.claims_amount_string
VALUES (1, 'POL-00001', '1250.50');

In [ ]:
# 4. After running pipeline 02_type_checks.py — compare valid vs quarantined records

for table, desc in [
    ("raw_claims",               "Raw input"),
    ("bronze_type_validated_claims", "DROP: type + value violations removed"),
    ("silver_valid_claims",      "Valid records ready for silver"),
    ("bronze_quarantine_claims", "Quarantined: failed type or value validation"),
]:
    try:
        cnt = spark.sql(f"SELECT COUNT(*) FROM trainer_demo.demo_09.{table}").collect()[0][0]
        print(f"  {table:<40} {cnt:>5}  {desc}")
    except:
        print(f"  {table:<40}   N/A  (run pipeline first)")

In [ ]:
%sql

-- Show sample quarantined records
SELECT
  claim_id,
  claim_amount,
  claim_type,
  status,
  quarantine_reason
FROM trainer_demo.demo_09.bronze_quarantine_claims
LIMIT 10;

## Detect and manage schema drift

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/implement-manage-data-quality-constraints-unity-catalog.detect-manage-schema-drift.png)

### Demo: Detect and Manage Schema Drift

**Scenario**: InsureCo's broker portal team adds a `risk_score` column to the policy feed
without notifying the data engineering team. How does the pipeline respond?

**Pipeline file**: `Demo/pipelines/09-data-quality/03_schema_drift.py`

**Steps**:
1. Show current schema of `raw_policies`
2. Simulate schema drift — add `risk_score` column and insert new data
3. Observe how strict streaming fails; rescued data pipeline keeps running
4. Show `mergeSchema` for automatic schema evolution
5. Demonstrate the quarantine pattern for structurally incomplete records

In [ ]:
%sql

-- 1. Inspect the current schema of raw_policies before drift
DESCRIBE TABLE trainer_demo.demo_09.raw_policies;

In [ ]:
%sql

-- 2. Simulate schema drift — broker adds a risk_score column
-- In production this would come from an upstream system change

ALTER TABLE trainer_demo.demo_09.raw_policies
ADD COLUMN IF NOT EXISTS risk_score DOUBLE;

INSERT INTO trainer_demo.demo_09.raw_policies
VALUES ('POL-DRIFT-001', 1234, 'Auto', '2025-01-01', '2026-01-01', 1200.00, 50000.00, 0.42);

DESCRIBE TABLE trainer_demo.demo_09.raw_policies;

In [ ]:
# 3. Rescue mode — the pipeline captures unexpected columns in _rescued_data
# After re-running 03_schema_drift.py, query the rescued data column

try:
    rescued = spark.sql("""
    SELECT policy_id, premium_amount, _rescued_data
    FROM trainer_demo.demo_09.bronze_policies_rescued
    WHERE _rescued_data IS NOT NULL
    LIMIT 10
    """)
    rescued.show(truncate=False)
except Exception:
    print("bronze_policies_rescued not yet created — run 03_schema_drift.py pipeline first")
    print()
    print("rescue mode: unexpected columns are stored as JSON in _rescued_data")
    print("  → pipeline keeps running without interruption")
    print("  → query _rescued_data to understand what new fields arrived")

In [ ]:
# 4. Schema evolution — automatically add new columns to the target table

spark.sql("""
CREATE TABLE IF NOT EXISTS trainer_demo.demo_09.policies_evolved
AS SELECT * FROM trainer_demo.demo_09.raw_policies WHERE 1=0
""")

# Write with mergeSchema=true — new columns are auto-added to target
(spark.table("trainer_demo.demo_09.raw_policies")
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("trainer_demo.demo_09.policies_evolved"))

print("Schema evolution write complete — risk_score column added automatically")
spark.sql("DESCRIBE TABLE trainer_demo.demo_09.policies_evolved").show(truncate=False)

In [ ]:
%sql

-- 5. Quarantine pattern — structural checks on raw_policies
-- Records missing required fields are routed to quarantine

SELECT
  COUNT(*) AS valid_count
FROM trainer_demo.demo_09.raw_policies
WHERE policy_id IS NOT NULL
  AND customer_id IS NOT NULL
  AND start_date IS NOT NULL
  AND end_date IS NOT NULL;

In [ ]:
%sql

SELECT
  COUNT(*) AS quarantine_count
FROM trainer_demo.demo_09.raw_policies
WHERE policy_id IS NULL
   OR customer_id IS NULL
   OR start_date IS NULL
   OR end_date IS NULL;

In [ ]:
%sql

SELECT
  policy_id,
  customer_id,
  start_date,
  end_date
FROM trainer_demo.demo_09.raw_policies
WHERE policy_id IS NULL
   OR customer_id IS NULL
   OR start_date IS NULL
   OR end_date IS NULL
LIMIT 10;

## Manage data quality with pipeline expectations

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/implement-manage-data-quality-constraints-unity-catalog.manage-data-quality-pipeline-expectations.png)

### Demo: Manage Data Quality with Pipeline Expectations

**Scenario**: InsureCo's silver-layer pipeline uses named expectations to enforce business
rules across all claim types. The pipeline UI shows pass/fail metrics for every rule.

**Pipeline file**: `Demo/pipelines/09-data-quality/04_pipeline_expectations.py`

**Steps**:
1. Walk through the three expectation components: **name**, **constraint**, **action**
2. Show reusable expectation dictionaries (`expect_all_or_drop`)
3. Compare `WARN` (monitor) vs `DROP` (clean) vs `FAIL` (critical integrity)
4. Run the pipeline and view **Data quality** metrics in the pipeline UI

**Viewing expectation metrics after pipeline run**:
> Jobs & Pipelines → select your pipeline → click a dataset (e.g. `silver_claims`) →
> open the **Data quality** tab in the right panel

In [ ]:
%sql

-- Show what would happen with WARN action (default) — all records flow through
-- Violations are counted but NOT blocked

SELECT
  COUNT(*) AS total_records,
  COUNT_IF(claim_id IS NULL) AS violations_claim_id_null,
  COUNT_IF(policy_id IS NULL) AS violations_policy_id_null,
  COUNT_IF(adjuster_id IS NULL) AS violations_no_adjuster,
  COUNT_IF(claim_amount <= 0) AS violations_bad_amount,
  COUNT_IF(claim_amount > 5000000) AS violations_amount_over_5m,
  COUNT_IF(status NOT IN ('Submitted', 'Under Review', 'Approved', 'Denied')) AS violations_invalid_status,
  COUNT_IF(claim_type NOT IN ('Auto', 'Property', 'Life', 'Health', 'Liberty')) AS violations_invalid_type
FROM trainer_demo.demo_09.raw_claims;

In [ ]:
# Reusable expectation dictionaries — how expect_all_or_drop works
# These mirror CLAIM_INTEGRITY_RULES and CLAIM_AMOUNT_RULES in 04_pipeline_expectations.py

integrity_pass = spark.sql("""
SELECT COUNT(*) FROM trainer_demo.demo_09.raw_claims
WHERE claim_id IS NOT NULL AND policy_id IS NOT NULL AND adjuster_id IS NOT NULL
""").collect()[0][0]

amount_pass = spark.sql("""
SELECT COUNT(*) FROM trainer_demo.demo_09.raw_claims
WHERE claim_amount > 0 AND claim_amount <= 5000000
  AND claim_date <= current_date() AND claim_date >= '2000-01-01'
""").collect()[0][0]

total = spark.sql("SELECT COUNT(*) FROM trainer_demo.demo_09.raw_claims").collect()[0][0]

print(f"Total records:                  {total}")
print(f"Pass CLAIM_INTEGRITY_RULES:     {integrity_pass}  (drop {total - integrity_pass})")
print(f"Pass CLAIM_AMOUNT_RULES:        {amount_pass}  (drop {total - amount_pass})")

In [ ]:
# FAIL action — check for approved claims with no adjuster (critical violation)
# These records would stop the silver_approved_claims pipeline

critical = spark.sql("""
SELECT claim_id, policy_id, status, adjuster_id, claim_amount
FROM trainer_demo.demo_09.raw_claims
WHERE status = 'Approved' AND adjuster_id IS NULL
""")

count = critical.count()
print(f"Approved claims with no adjuster (would trigger FAIL action): {count}")
if count > 0:
    critical.show(10, truncate=False)

In [ ]:
# After running pipeline 04_pipeline_expectations.py — compare all output tables

tables = [
    ("raw_claims",                   "Raw input"),
    ("silver_claims",                "DROP all INTEGRITY + AMOUNT rules"),
    ("silver_claims_monitored",      "WARN  all records kept, violations logged"),
    ("silver_approved_claims",       "FAIL  on missing adjuster for approved"),
    ("silver_high_value_auto_claims","WARN  high-value auto + business rule"),
]

print(f"{'Table':<42} {'Count':>6}  Description")
print("-" * 80)
for table, desc in tables:
    try:
        cnt = spark.sql(f"SELECT COUNT(*) FROM trainer_demo.demo_09.{table}").collect()[0][0]
        print(f"  {table:<40} {cnt:>6}  {desc}")
    except:
        print(f"  {table:<40}    N/A  (run pipeline first)")

### Viewing Expectation Metrics in the Pipeline UI

After running the pipeline:

1. Navigate to **Jobs & Pipelines** in the left sidebar
2. Select your pipeline by name
3. Click on a dataset (e.g., `silver_claims`) in the pipeline graph
4. Open the **Data quality** tab in the right panel

You will see for each expectation:
- **Name** — e.g., `positive_amount`, `valid_claim_type` (descriptive names matter!)
- **Pass count** — records that satisfied the constraint
- **Fail count** — records that violated the constraint
- **Action taken** — warn / drop / fail

Well-named expectations make the monitoring dashboard immediately actionable.
Metrics accumulate across pipeline runs, giving you trend visibility over time.